# Auslan → English: training on Colab

Runs after `colab_setup.ipynb` has extracted the poses and its gate passed.
That notebook leaves three things in `auslan_work/` on Drive, and this one
trains from them:

- `manifest.jsonl`: which clips exist, their split and their English text
- `pose/chunk_*.tar`: the extracted keypoints
- `excluded.txt`: the clips the gate found mirrored (131 of 25,109), which
  training leaves out

Run the sections in order: 1–6 set up, 7 re-scores finished runs under other
decoding settings, 8 trains the runs chosen in section 6, 9 compares
everything. **In a new session after a disconnect, run 1–6 again and then 8,**
which continues from the last checkpoint on Drive. 7 and 9 can be re-run at
any time: they skip what is already done.

## 1. Runtime check

Training the real model needs a GPU. On CPU, one epoch would take days.

In [1]:
import subprocess
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip() or 'NO GPU')
DEVICE = 'cuda'

NVIDIA A100-SXM4-40GB, 40960 MiB


## 2. Dependencies

Colab already has torch, transformers and sentencepiece. Uni-Sign's model
code also needs `einops`, and the BLEU score needs `sacrebleu`.

In [2]:
!pip -q install einops sacrebleu
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__,
      '| cuda', torch.cuda.is_available())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 10.7 MB/s eta 0:00:00
torch 2.11.0+cu128 | transformers 5.16.1 | cuda True


## 3. Mount Drive and find the extraction's outputs

In [4]:
import os
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive'

WORK = f'{DRIVE}/auslan_work'
MANIFEST = f'{WORK}/manifest.jsonl'
EXCLUDE = f'{WORK}/excluded.txt'
for path, made_by in [(MANIFEST, 'section 6'), (EXCLUDE, 'the gate, section 10')]:
    if not os.path.exists(path):
        raise SystemExit(f'{path} is missing. It is written by {made_by} of '
                         'colab_setup.ipynb; run that first.')
print('work dir:', WORK)

Mounted at /content/drive
work dir: /content/drive/MyDrive/auslan_work


## 4. The code

The same `unisign/` folder `colab_setup.ipynb` uses: a `unisign/` folder on
Drive, or `unisign_code.tar.gz`. Upload the current version whenever the code
changes locally.

In [5]:
import glob, shutil, sys, tarfile

CODE = '/content/unisign'

def locate(pattern_parts):
    for root in [DRIVE, '/content']:
        for prefix in ['', '*/', '*/*/']:
            hits = glob.glob(os.path.join(root, prefix, *pattern_parts))
            if hits:
                return hits[0]
    return None

src = locate(['unisign', 'spec.py'])
if src:
    src = os.path.dirname(src)
else:
    tarball = locate(['unisign_code.tar.gz'])
    if tarball is None:
        raise SystemExit('Neither a unisign/ folder nor unisign_code.tar.gz was '
                         'found on Drive. Upload one.')
    with tarfile.open(tarball) as tf:
        tf.extractall('/content/_code')
    src = '/content/_code/unisign'
if os.path.abspath(src) != CODE:
    shutil.rmtree(CODE, ignore_errors=True)
    shutil.copytree(src, CODE)
sys.path.insert(0, CODE)
import spec
print('code from', src)
print('spec fingerprint', spec.SCHEMA_FINGERPRINT, '(the extraction used bc3bb2df0f22948d)')
assert spec.SCHEMA_FINGERPRINT == 'bc3bb2df0f22948d', \
    'spec.py differs from the one the poses were extracted with'

code from /content/drive/MyDrive/unisign
spec fingerprint bc3bb2df0f22948d (the extraction used bc3bb2df0f22948d)


## 5. Model: Uni-Sign code, mT5 and the pre-trained weights

Everything is downloaded from its public source, pinned to exactly the
versions used locally, and checked by sha256, so the model trained here is the
one that was verified. About 3.5 GB, a few minutes.

`INIT_CKPT` picks the starting weights. Each choice gets its own run folder,
so runs from different starting points never mix:

- `csl_stage1_weight.pth`: pose-only pre-trained base (CSL-News), the clean
  transfer starting point. **Start here.**
- `how2sign_pose_only_slt.pth` / `openasl_pose_only_slt.pth`: already
  fine-tuned to produce English from ASL. They may transfer better to an
  English target. Worth running as comparisons once the first run works.

In [6]:
import hashlib, subprocess
from huggingface_hub import hf_hub_download, snapshot_download

INIT_CKPT = 'csl_stage1_weight.pth'

REPO_DIR = '/content/Uni-Sign'
REPO_COMMIT = 'eed438bcb49e30405cd6ccdfcccca330c134e830'
MT5_DIR = f'{REPO_DIR}/pretrained_weight/mt5-base'
MT5_REVISION = '2eb15465c5dd7f72a8f7984306ad05ebc3dd1e1f'
UNISIGN_REVISION = 'eab251b7fe7e8521afc0e67be98add670ea40a0d'
SHA256 = {
    'csl_stage1_weight.pth':      '3c81cf4a087e9e81581e57a2f33f8f0acf87b518a1ada540a660a76cdb144ced',
    'how2sign_pose_only_slt.pth': '1bfd5f3312f04e4736f0a52f4ef9535916e6de9676a2a0d00c708748683fb00d',
    'openasl_pose_only_slt.pth':  'f836ea66bc837bbe6ed717a4b9bece87875f03ef96d4bf1092ca3dd767982798',
    'mt5-base/pytorch_model.bin': '180573b534144580f04af026da62bf71bc976ee1b7eb311b8945e2fefde8d614',
}

if not os.path.isdir(f'{REPO_DIR}/.git'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/ZechengLi19/Uni-Sign.git',
                    REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'checkout', '-q', REPO_COMMIT], check=True)

# PyTorch weights and tokenizer only: the TF and Flax copies are 4.7 GB of the
# same weights in formats nothing here reads.
snapshot_download('google/mt5-base', revision=MT5_REVISION, local_dir=MT5_DIR,
                  allow_patterns=['*.json', '*.model', 'pytorch_model.bin'])
CKPT = hf_hub_download('ZechengLi19/Uni-Sign', INIT_CKPT, revision=UNISIGN_REVISION,
                       local_dir='/content/checkpoints')

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 24), b''):
            h.update(block)
    return h.hexdigest()

for name, path in [(INIT_CKPT, CKPT),
                   ('mt5-base/pytorch_model.bin', f'{MT5_DIR}/pytorch_model.bin')]:
    if sha256(path) != SHA256[name]:
        raise SystemExit(f'{name}: sha256 mismatch -- delete {path} and re-run this cell')
    print(f'OK  {name}')
print('Uni-Sign code at', REPO_COMMIT[:7])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

csl_stage1_weight.pth: reconstructing file:   0%|          |  0.00B / 1.19GB            

csl_stage1_weight.pth: downloading bytes:           |  0.00B            

OK  csl_stage1_weight.pth
OK  mt5-base/pytorch_model.bin
Uni-Sign code at eed438b


## 6. The poses and the run config

Unpacks the extracted keypoints from Drive onto local disk: reading 25k
small files through the Drive mount during training would be far slower. Then
writes this run's config to Drive. It is the repo's `arm_a.yaml` with the
paths for this machine, the real model instead of the test one, and
`data.exclude` pointing at the gate's list.

The config is regenerated identically every session. That matters: `--resume`
refuses to continue a run whose config changed.

`LR` and `SEEDS` choose what section 8 trains. The first baseline, 3e-5 with
seeds 0 and 1, is always listed too, as the reference, under the run names it
was trained with; a run at another rate carries the rate in its name
(`…__lr1e-04`). `DECODES` lists the decoding settings sections 7–9 score every
run under. This cell also defines the helpers sections 7 and 8 launch
`train.py` with, so it has to run in every session.

In [7]:
import json, yaml

POSE_LOCAL = '/content/pose'
os.makedirs(POSE_LOCAL, exist_ok=True)
rows = [json.loads(l) for l in open(MANIFEST)]
if len(glob.glob(f'{POSE_LOCAL}/*.npz')) < len(rows):
    chunks = sorted(glob.glob(f'{WORK}/pose/chunk_*.tar'))
    print(f'unpacking {len(chunks)} chunk(s) from Drive ...')
    for c in chunks:
        with tarfile.open(c) as tf:
            tf.extractall(POSE_LOCAL)
present = {os.path.basename(p)[:-4] for p in glob.glob(f'{POSE_LOCAL}/*.npz')}
missing = [r['uid'] for r in rows if r['uid'] not in present]
excluded = {l.strip() for l in open(EXCLUDE) if l.strip() and not l.startswith('#')}
print(f'{len(rows)} clips in the manifest | {len(present)} poses on disk | '
      f'{len(missing)} missing | {len(excluded)} excluded by the gate')
if missing:
    raise SystemExit(f'{len(missing)} clips have no pose, e.g. {missing[:3]}. '
                     'Finish the extraction in colab_setup.ipynb first.')

import copy, signal, subprocess, sys

ARM = 'arm_a'
# The first baseline trained at 3e-5 with seeds 0 and 1. Both runs are finished
# and stay exactly as they are, as the reference. The learning-rate experiment
# changes the rate and nothing else, and screens it with one seed: seed 0, which
# also gives it baseline seed 0's batch order, so the two differ in the rate only.
BASELINE_LR, BASELINE_SEEDS = 3e-5, [0, 1]
LR = 1e-4
SEEDS = [0]

BASE = f'{ARM}__{INIT_CKPT.rsplit(".", 1)[0]}'
base_cfg = yaml.safe_load(open(f'{CODE}/configs/{ARM}.yaml'))

def make_run(lr, seed):
    # The 3e-5 runs keep the names they were trained under; others carry the rate.
    run = BASE if lr == BASELINE_LR else f'{BASE}__lr{lr:.0e}'
    if seed != 0:
        run += f'__seed{seed}'
    c = copy.deepcopy(base_cfg)
    c['seed'] = seed
    c['optim']['lr'] = lr
    c['data'].update(manifest=MANIFEST, npz_dir=POSE_LOCAL, exclude=EXCLUDE)
    c['backend'] = {'name': 'unisign', 'checkpoint': CKPT, 'repo': REPO_DIR,
                    'mt5_path': MT5_DIR, 'num_beams': 5, 'max_new_tokens': 100}
    c['output_dir'] = f'{WORK}/runs/{run}'
    path = f'{WORK}/train_configs/{run}.yaml'
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w') as f:
        yaml.safe_dump(c, f, sort_keys=False)
    return run, path, c

# Every run the notebook knows, keyed (lr, seed): the baseline to compare
# against, and the runs section 8 trains.
ALL_RUNS = {(lr, s): make_run(lr, s)
            for lr, seeds in ((BASELINE_LR, BASELINE_SEEDS), (LR, SEEDS)) for s in seeds}
RUNS = {s: ALL_RUNS[(LR, s)] for s in SEEDS}
print()
for (lr, s), (run, path, c) in ALL_RUNS.items():
    role = 'baseline' if lr == BASELINE_LR else 'to train'
    print(f'lr {lr:.0e} seed {s} ({role}): {run}\n  output {c["output_dir"]}')
cfg = RUNS[SEEDS[0]][2]                 # shown below
print()
print(yaml.safe_dump({k: cfg[k] for k in ('data', 'backend', 'optim')}, sort_keys=False))

# Decoding settings sections 7-9 score every finished run under, besides the
# plain beam search each run is evaluated with when it finishes. Against the
# loops ("a bit of a bit of ...") and stock sentences the baseline fell into.
DECODES = {
    'nr3':      {'no_repeat_ngram_size': 3},
    'nr3_rp12': {'no_repeat_ngram_size': 3, 'repetition_penalty': 1.2},
}

def run_train(cfg_path, extra, log_path=None):
    # Streams the output as it arrives. Stop sends train.py a Ctrl-C, which
    # makes it save a checkpoint before exiting, instead of killing it mid-step.
    cmd = [sys.executable, '-u', 'train.py', '--config', cfg_path, '--device', DEVICE] + extra
    log = open(log_path, 'a') if log_path else None
    p = subprocess.Popen(cmd, cwd=CODE, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    def pump():
        for line in p.stdout:
            print(line, end='', flush=True)
            lines.append(line)
            if log:
                log.write(line)
                log.flush()
    try:
        pump()
    except KeyboardInterrupt:
        print('\nStop pressed: letting train.py save a checkpoint, then exit ...', flush=True)
        p.send_signal(signal.SIGINT)
        pump()
    p.wait()
    if log:
        log.close()
    return p.returncode, lines

def finished(c):
    return os.path.exists(f"{c['output_dir']}/metrics.json")

def eval_dir(c, tag):
    # 'plain' is the evaluation every run does when it finishes.
    return c['output_dir'] if tag == 'plain' else f"{c['output_dir']}/eval_{tag}"

def evaluate_decodes(run, path, c, decodes):
    """Score a finished run under each {tag: options}, skipping those done."""
    for tag, opts in decodes.items():
        if os.path.exists(f'{eval_dir(c, tag)}/metrics.json'):
            print(f'{run} [{tag}]: already scored')
            continue
        print(f'\n=== {run} [{tag}] {opts or "plain"}\n', flush=True)
        sets = [f'decode.{k}={v}' for k, v in opts.items()]
        rc, _ = run_train(path, ['--eval-only', '--eval-tag', tag]
                          + (['--set', *sets] if sets else []))
        if rc != 0:
            raise RuntimeError(f'scoring {run} [{tag}] failed (exit {rc}). Re-running '
                               'the cell skips what is already scored.')


unpacking 216 chunk(s) from Drive ...


/tmp/ipykernel_4145/3878018263.py:11: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(POSE_LOCAL)


25109 clips in the manifest | 25109 poses on disk | 0 missing | 131 excluded by the gate

seed 0: arm_a__csl_stage1_weight
  config /content/drive/MyDrive/auslan_work/train_configs/arm_a__csl_stage1_weight.yaml
  output /content/drive/MyDrive/auslan_work/runs/arm_a__csl_stage1_weight
seed 1: arm_a__csl_stage1_weight__seed1
  config /content/drive/MyDrive/auslan_work/train_configs/arm_a__csl_stage1_weight__seed1.yaml
  output /content/drive/MyDrive/auslan_work/runs/arm_a__csl_stage1_weight__seed1

data:
  manifest: /content/drive/MyDrive/auslan_work/manifest.jsonl
  npz_dir: /content/pose
  max_length: 256
  gloss_text_mode: strip_paren
  exclude: /content/drive/MyDrive/auslan_work/excluded.txt
backend:
  name: unisign
  checkpoint: /content/checkpoints/csl_stage1_weight.pth
  repo: /content/Uni-Sign
  mt5_path: /content/Uni-Sign/pretrained_weight/mt5-base
  num_beams: 5
  max_new_tokens: 100
optim:
  lr: 3.0e-05
  weight_decay: 0.01
  batch_size: 8
  epochs: 20
  warmup_frac: 0.05
  gr

## 7. Decoding: re-score the finished runs

The baseline's outputs loop ("it is a bit of a bit of …" in 64–77% of the News
outputs) and fall back on a few stock sentences. This scores every finished run
again under each setting in `DECODES`, from its final weights and without
training: `train.py --eval-only` writes `eval_<tag>/` into the run folder and
leaves the run's own results alone.

The first time, it also scores baseline seed 0 with plain decoding through the
same path. That has to reproduce the run's own scores exactly, and checks that
the evaluation loads the right weights. Each evaluation decodes the 1,492
validation clips; finished ones are skipped, so re-running the cell is cheap.

In [ ]:
# Once: plain decoding through --eval-only must give baseline seed 0's own scores.
run, path, c = ALL_RUNS[(BASELINE_LR, 0)]
evaluate_decodes(run, path, c, {'check_plain': {}})
own = json.load(open(f"{c['output_dir']}/metrics.json"))
again = json.load(open(f"{eval_dir(c, 'check_plain')}/metrics.json"))
diff = max(abs(own[g][k] - again[g][k]) for g in own
           for k in ('BLEU-1', 'BLEU-2', 'BLEU-3', 'BLEU-4', 'ROUGE-L'))
if diff > 0.01:
    raise RuntimeError(f'--eval-only with plain decoding is {diff} away from the run\'s '
                       'own scores. Do not trust the decoding comparison until that is '
                       'explained; paste the output back.')
print(f'check passed: --eval-only reproduces {run}\'s own scores (largest difference {diff})\n')

for (lr, s), (run, path, c) in ALL_RUNS.items():
    if finished(c):
        evaluate_decodes(run, path, c, DECODES)
    else:
        print(f'{run}: not trained yet; section 8 scores it when it finishes')
print('\nDone. Section 9 prints the comparison.')

## 8. Train

Trains every seed in `SEEDS` at `LR` (section 6). Only the learning rate differs
from the baseline: same data, same starting weights, same 20 epochs and
schedule, and seed 0 means the same batch order as baseline seed 0. About 5
hours per seed on the A100.

Saves a checkpoint to Drive every 30 minutes and at every epoch end. Only the
newest two are kept, about 7 GB each. The output also goes to `train.log` in
the run folder.

**Stopping is safe.** Pressing Stop saves a checkpoint first. If the runtime
disappears, at most the last 30 minutes are lost. Either way, re-run 1–6 and
then this cell. It skips a run that has finished and continues the one in
progress from its last checkpoint. It always runs with `--resume`, which also
works on the very first run.

At the end of each run, `train.py` evaluates on the validation split with plain
beam search, as the baseline was, and writes `metrics.json` and
`predictions.jsonl` into the run folder. The cell then scores it under each
`DECODES` setting as well. The final weights are `checkpoint.pt` there.

In [8]:
for seed in SEEDS:
    run, path, c = RUNS[seed]
    out = c['output_dir']
    if finished(c):
        print(f'seed {seed}: already finished ({out})')
    else:
        os.makedirs(out, exist_ok=True)
        print(f'\n=== lr {LR:.0e} seed {seed}: {run} -> {out}\n')
        rc, _ = run_train(path, ['--resume'], log_path=f'{out}/train.log')
        if rc == 130:
            print('\nSTOPPED with a checkpoint saved. Re-run this cell to continue.')
            break
        if rc != 0:
            print(f'\ntrain.py exited with code {rc}. The newest checkpoint on Drive is '
                  'intact; paste the output above before re-running.')
            break
        print(f'\nseed {seed} FINISHED. Final weights: {out}/checkpoint.pt')
    evaluate_decodes(run, path, c, DECODES)
print('\nSection 9 prints the comparison.')

seed 0: already finished (/content/drive/MyDrive/auslan_work/runs/arm_a__csl_stage1_weight)

=== seed 1: arm_a__csl_stage1_weight__seed1 -> /content/drive/MyDrive/auslan_work/runs/arm_a__csl_stage1_weight__seed1

arm=arm_a device=cuda out=/content/drive/MyDrive/auslan_work/runs/arm_a__csl_stage1_weight__seed1
train=21998 val=1492
left out 119 clips listed in /content/drive/MyDrive/auslan_work/excluded.txt

Loading weights: 100%|██████████| 284/284 [00:00<00:00, 21856.96it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[load] missing=0 unexpected=0
trainable groups=['decoder', 'pose_encoder', 'temporal'] 587.75M / 587.75M (100.0%)
    decoder         582.40M  lr=3.00e-05
    pose_encoder      0.41M  lr=3.00e-05
    temporal          4.94M  lr=3

## 9. Compare

Every finished run under every decoding setting, per subset, recomputed from the
saved predictions. `distinct%` is the share of different outputs, `looping%` the
share that repeat a word trigram, `words` the mean output length (the
references average about 5 words in Communication and 16 in News).

Compare rows with the same decoding only. The baseline's two seeds give the
noise floor: a change counts only if, per subset, it lands above the baseline's
range by more than the gap between its seeds. On Communication under plain
decoding that means BLEU-4 above about 4.7. News BLEU-4 is too close to 0 to
decide anything; look at its ROUGE-L and `looping%` instead. One seed is a
screen: a clear result stands, a marginal one needs a second seed.

In [ ]:
from evaluate import evaluate_records

COLS = [('BLEU-1', 'BLEU-1'), ('BLEU-4', 'BLEU-4'), ('ROUGE-L', 'ROUGE-L'),
        ('unique_hyps', 'distinct%'), ('looping', 'looping%'), ('hyp_len', 'words')]
TAGS = ['plain', *DECODES]
scores = {}
for (lr, s), (run, path, c) in ALL_RUNS.items():
    for tag in TAGS:
        d = eval_dir(c, tag)
        if os.path.exists(f'{d}/metrics.json'):
            recs = [json.loads(l) for l in open(f'{d}/predictions.jsonl')]
            scores[(lr, s, tag)] = evaluate_records(recs)
if not scores:
    raise SystemExit('Nothing is scored yet.')

for g in sorted({g for m in scores.values() for g in m}):
    n = next(m[g]['n'] for m in scores.values() if g in m)
    print(f'\n{g}  (n={n})')
    print(f"  {'lr':>6} {'seed':>4}  {'decode':<9}" + ''.join(f'{h:>10}' for _, h in COLS))
    for tag in TAGS:
        rows = [(lr, s, m[g]) for (lr, s, t), m in scores.items() if t == tag and g in m]
        for lr, s, m in rows:
            print(f'  {lr:>6.0e} {s:>4}  {tag:<9}' + ''.join(f'{m[k]:>10.2f}' for k, _ in COLS))
        base = [m['BLEU-4'] for lr, s, m in rows if lr == BASELINE_LR]
        if len(base) > 1:
            lo, hi = min(base), max(base)
            print(f"  {'':>6} {'':>4}  {'':<9}  baseline BLEU-4 {lo:.2f}-{hi:.2f}, gap "
                  f"{hi - lo:.2f}: a change needs more than {hi + (hi - lo):.2f}")
        if rows:
            print()
print('Compare rows with the same decoding only. Do not average the two subsets.')

## If the session dies

Nothing is lost except the time since the last checkpoint, at most 30 minutes.

1. Reconnect and run sections **1–6** in order. Section 5 downloads the model
   again, which takes a few minutes. Section 6 unpacks the poses again.
2. Run section **8**. It skips any run that has finished, prints
   `resumed from ckpt_step…` for the one in progress, and continues. If it
   says `starting fresh` for a run that was already under way, press Stop
   and check that the run folder on Drive still has its `checkpoints/`.
3. Sections 7 and 9 can be run whenever you like; they skip what is done.

Do not change `INIT_CKPT`, the config or `excluded.txt` while a run is under
way. `--resume` checks that the run is the same one and refuses otherwise,
rather than silently mixing two runs. Changing `LR` or `SEEDS` only selects
other runs.